# Module 2.1: Tokenization & Embeddings (The Building Blocks)

In the previous module, we learned how Transformers do math. But text is made of letters, not numbers. We must translate human language into a format the network can compute. This process has two stages: **Tokenization** (chunking) and **Embedding** (vectorizing).

## 1. Tokenization: Text to Chunks

### The Concept
We could split text by single characters ('a', 'b', 'c'), or by whole words ('apple', 'banana'). Modern LLMs use a middle-ground called **Sub-word Tokenization** (like Byte-Pair Encoding or BPE). Common words stay whole ('apple'), but rare words get split into common chunks ('un', 'believ', 'able').

### Why?
1. **Character-level** is too dense: The word 'apple' would require 5 steps of computation just to understand one piece of fruit. The model loses context over long distances.
2. **Word-level** is too sparse: The English dictionary is essentially infinite (e.g., 'Google', 'Googling', 'Googled'). The vocabulary matrix would be too huge to fit in GPU memory.
3. **Sub-word (BPE)** is the sweet spot: It handles unknown words gracefully by breaking them down, while keeping the total vocabulary size manageable (usually around 50k to 100k tokens).

In [1]:
import tiktoken

# GPT-4's tokenizer (cl100k_base)
encoder = tiktoken.get_encoding("cl100k_base")

text = "Transformers are unbelievably fast!"
tokens = encoder.encode(text)

print(f"Raw String: '{text}'")
print(f"Token IDs (Integers): {tokens}")

print("\nBreaking it down:")
for token_id in tokens:
    print(f"{token_id} -> '{encoder.decode([token_id])}'")

Raw String: 'Transformers are unbelievably fast!'
Token IDs (Integers): [9140, 388, 527, 40037, 89234, 5043, 0]

Breaking it down:
9140 -> 'Transform'
388 -> 'ers'
527 -> ' are'
40037 -> ' unbelie'
89234 -> 'vably'
5043 -> ' fast'
0 -> '!'


## 2. The Lookup Table: Input IDs to Vectors

### The Problem
We now have integers representing words (e.g., ID `45` and `12933`). But Neural Networks cannot do calculus or $\sum a_i b_i$ math on the raw integer `12933`. Furthermore, the number `12933` has no geometric meaning—it's not "mathematically closer" to `12934` in meaning. We need to translate these integers into a continuous mathematical space.

### The Analogy (What is an "Embedding Dimension"?)
Imagine trying to describe a person using just an ID number (e.g., Person 45). That tells you nothing about who they are. Instead, you could describe them using 3 **dimensions** (traits): `[Friendliness, Intelligence, Athleticism]`.

An **Embedding Dimension** is exactly this. In LLMs, we choose a number (e.g., 256) and declare: *"Every word in our dictionary will be described by 256 different unknown mathematical traits."* The neural network will figure out what those 256 traits mean during training. One dimension might learn to track 'gender', another might track 'plurality' (cat vs cats), and another might track 'royalty' (King vs Man).

### The Math
We use `torch.nn.Embedding`. If our vocabulary size is $V$ (50,000 words) and our chosen embedding dimension is $D$ (256 traits), the Embedding layer is just a giant matrix of size $(V 	imes D)$. Giving it the ID `45` simply plucks out row number `45`, returning a vector of 256 numbers.

### Why do we need it?
By replacing a meaningless integer with a dense vector of traits, the model can place the word in a semantic "concept space". Words with similar meanings will naturally have similar embedding vectors. As the model trains using Backpropagation, it physically adjusts these embedding weights, moving similar words closer together in this mathematical space!

In [2]:
import torch
import torch.nn as nn

# Setup: A vocabulary of 50,000 possible tokens. Each token gets a 256-dimension vector.
vocab_size = 50000
embedding_dim = 256

token_embedder = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

# Create a fake "batch" of data. 
# Shape: (Batch_Size, Sequence_Length) -> (2 sentences, 4 words each)
input_ids = torch.tensor([
    [45, 102, 900, 3],   # Sentence 1
    [8, 12933, 4, 1]     # Sentence 2
])

# Translation step!
embedded_output = token_embedder(input_ids)

print("Dimensionality Tracking 🔥")
print(f"Input Shape: {input_ids.shape} -> (Batch, Seq_Length)")
print(f"Output Shape: {embedded_output.shape} -> (Batch, Seq_Length, Embed_Dim)")

Dimensionality Tracking 🔥
Input Shape: torch.Size([2, 4]) -> (Batch, Seq_Length)
Output Shape: torch.Size([2, 4, 256]) -> (Batch, Seq_Length, Embed_Dim)


## 3. Under the Hood (One-Hot Encoding)

### The Concept
A "lookup table" sounds like standard programming, not Calculus. How does Backpropagation push gradients through a table lookup?

### The Math
Mathematically, selecting a row from a matrix is exactly equivalent to performing a **Matrix Multiplication** between the embedding matrix and a **One-Hot Encoded** vector (a vector of all zeros except for a single '1' at the target index).

### Why?
By framing the lookup as a Matrix Multiplication, the entire operation becomes differentiable. The gradients from the Error/Loss can flow seamlessly back into the Embedding Matrix, adjusting the values of the vectors so the model learns semantic relationships.

In [3]:
import torch.nn.functional as F

# Let's use a tiny vocabulary of 5 words, and a dimension of 3 (for easy printing)
tiny_vocab = 5
tiny_dim = 3

# Our raw embedding matrix (weights)
emb_layer = nn.Embedding(tiny_vocab, tiny_dim)

# Let's say we want to embed the integer identity '2'
word_id = torch.tensor([2])

# --- METHOD 1: The Fast Lookup ---
lookup_result = emb_layer(word_id)

# --- METHOD 2: The Math Way (One-Hot + MatMul) ---
# 1. Create a One-Hot vector: [0., 0., 1., 0., 0.]
one_hot = F.one_hot(word_id, num_classes=tiny_vocab).float()

# 2. Multiply by the embedding weights matrix
matmul_result = torch.matmul(one_hot, emb_layer.weight)

print("Lookup Method Output: ", lookup_result.detach())
print("MatMul Method Output: ", matmul_result.detach())
print("\nAre they mathematically identical? ->", torch.allclose(lookup_result, matmul_result))

Lookup Method Output:  tensor([[-0.1425, -0.3570, -1.0851]])
MatMul Method Output:  tensor([[-0.1425, -0.3570, -1.0851]])

Are they mathematically identical? -> True
